In [ ]:
import os
import sys
import json
import csv
import gc
import math
import time
import hashlib
import datetime

import numpy as np
from scipy.special import erfc, gammaincc
from scipy.stats import norm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ALPHA = 0.01
N_RAW_SINGLE = 1_000_000       # dlugosc ciagu surowego do testow pojedynczych
K_SEQ = 100                    # liczba podciagow w analizie wielosekwencyjnej
N_SUB_RAW = 12_000
N_SUB_XOR = 6_000
DPI = 300
T0 = time.time()

FILES_IN = {
    "sq1": "sq1_public.txt",
    "sq2": "sq2_secret.txt",
    "sq3": "sq3_control.txt",
    "ckpt": "eqrng_checkpoint.json",
}


def log(msg=""):
    print(msg, flush=True)


def hdr(title):
    log("\n" + "=" * 78)
    log(title)
    log("=" * 78)


def json_default(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    raise TypeError(f"Nieserializowalny typ: {type(o)}")


# Sekcja 1: Wczytywanie danych i weryfikacja pliku checkpointu z Etapu 1

def load_bits(path):
    """Wczytuje ciag znakow '0'/'1' (biale znaki ignorowane) jako np.uint8."""
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Brak pliku wejsciowego: {path}")
    with open(path, "rb") as f:
        raw = np.frombuffer(f.read(), dtype=np.uint8)
    valid = (raw == 48) | (raw == 49)
    white = np.isin(raw, (9, 10, 13, 32))
    if np.any(~(valid | white)):
        raise ValueError(f"Plik {path} zawiera znaki inne niz '0', '1' i biale znaki.")
    return (raw[valid] - 48).astype(np.uint8)


def load_checkpoint(path):
    if not os.path.isfile(path):
        log(f"[info] Brak {path} - pomijam (plik opcjonalny).")
        return None
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:  # noqa
        log(f"[ostrzezenie] Nie udalo sie wczytac {path}: {e}")
        return None


def flatten_numeric(d, prefix=""):
    out = {}
    if isinstance(d, dict):
        for k, v in d.items():
            out.update(flatten_numeric(v, f"{prefix}{k}."))
    elif isinstance(d, (int, float)) and not isinstance(d, bool):
        out[prefix.rstrip(".")] = d
    return out


def checkpoint_crosscheck(ckpt, n_bits):
    if ckpt is None:
        return {"present": False}
    flat = flatten_numeric(ckpt)
    info = {"present": True, "numeric_keys": list(flat.keys())[:40]}

    vt = ckpt.get("shots_processed")
    vv = ckpt.get("valid_shots")
    ve = ckpt.get("error_shots")

    if vt is None and vv is not None and ve is not None:
        vt = int(vv) + int(ve)

    log(f"[checkpoint] klucze liczbowe: {list(flat.keys())[:12]}")
    if vv is not None:
        ok = int(vv) == n_bits
        info["valid_matches_sq1_length"] = ok
        log(f"[checkpoint] poprawne = {vv} vs dlugosc S_Q1 = {n_bits} -> {'OK' if ok else 'NIEZGODNOSC'}")
    if vt is not None and vv is not None and float(vt) > 0:
        qber = 1.0 - float(vv) / float(vt)
        info["qber_recomputed"] = qber
        log(f"[checkpoint] QBER przeliczony = {100*qber:.4f}% (wszystkie={vt})")
    return info


# MODUL 1: Regula XOR, niezaleznosc par ciagow i kasowanie S_Q3

def pair_metrics(a, b):
    n = len(a)
    ham = float(np.count_nonzero(a != b)) / n
    af, bf = a.astype(np.float64), b.astype(np.float64)
    sa, sb = af.std(), bf.std()
    r = float(np.mean((af - af.mean()) * (bf - bf.mean())) / (sa * sb)) if sa > 0 and sb > 0 else float("nan")
    joint = np.bincount(a.astype(np.intp) * 2 + b.astype(np.intp), minlength=4).astype(np.float64) / n
    px = np.array([joint[0] + joint[1], joint[2] + joint[3]])
    py = np.array([joint[0] + joint[2], joint[1] + joint[3]])
    mi = 0.0
    for i in range(2):
        for j in range(2):
            pij = joint[i * 2 + j]
            if pij > 0:
                mi += pij * math.log2(pij / (px[i] * py[j]))
    mi = max(mi, 0.0)
    g = 2.0 * n * math.log(2) * mi          # statystyka G ~ chi2(1) przy niezaleznosci
    p_ind = float(gammaincc(0.5, g / 2.0))
    return {
        "hamming_norm": ham,
        "pearson_r": r,
        "mutual_info_bits": mi,
        "G_stat": g,
        "independence_p_value": p_ind,
    }


def run_module1(sq1, sq2, holder, ckpt, shred_file):
    hdr("MODUL 1: weryfikacja regul XOR, niezaleznosc bitowa, kasowanie S_Q3")
    sq3 = holder["sq3"]
    if not (len(sq1) == len(sq2) == len(sq3)):
        raise ValueError(f"Rozne dlugosci ciagow: {len(sq1)}, {len(sq2)}, {len(sq3)}")
    n = len(sq1)
    res = {"n_bits": n}
    res["checkpoint"] = checkpoint_crosscheck(ckpt, n)

    mism = {
        "S_Q1 ^ S_Q2 == S_Q3": int(np.count_nonzero((sq1 ^ sq2) != sq3)),
        "S_Q1 ^ S_Q3 == S_Q2": int(np.count_nonzero((sq1 ^ sq3) != sq2)),
        "S_Q2 ^ S_Q3 == S_Q1": int(np.count_nonzero((sq2 ^ sq3) != sq1)),
    }
    res["xor_mismatches"] = mism
    for k, v in mism.items():
        log(f"  Regula {k}: niezgodnosci = {v}  -> {'OK' if v == 0 else 'BLAD!'}")
    if any(v != 0 for v in mism.values()):
        log("  [OSTRZEZENIE] Reguly XOR naruszone - dane z Etapu 1 sa niespojne!")

    pairs = {"S_Q1 vs S_Q2": (sq1, sq2), "S_Q1 vs S_Q3": (sq1, sq3), "S_Q2 vs S_Q3": (sq2, sq3)}
    res["pairs"] = {}
    log("\n  Para            | Hamming | Pearson r  | I(X;Y) [bit] | p(niezal.)")
    for name, (a, b) in pairs.items():
        m = pair_metrics(a, b)
        res["pairs"][name] = m
        log(f"  {name:<15} | {m['hamming_norm']:.5f} | {m['pearson_r']:+.6f} | "
            f"{m['mutual_info_bits']:.3e}  | {m['independence_p_value']:.4f}")
    log("  Uwaga: parami ciagi sa niezalezne, ale trojka (S_Q1,S_Q2,S_Q3) jest deterministycznie"
        "\n  zwiazana regula XOR - dlatego usuniecie S_Q3 jest warunkiem bezpieczenstwa protokolu.")

    # Krok 7 protokolu: kasowanie S_Q3
    sq3.fill(0)
    assert not sq3.any()
    del sq3
    holder.pop("sq3")
    gc.collect()
    log("\n  [Krok 7 EQRNG] S_Q3 wyzerowany w RAM i usuniety (del sq3).")
    log("  Od tego momentu S_Q2 nie moze byc odtworzony z publicznego S_Q1 przy pomocy S_Q3,")
    log("  o ile S_Q3 nie istnieje juz nigdzie indziej (dysk, logi, dostawca chmury).")
    res["sq3_erased_from_ram"] = True
    path3 = os.path.join(os.getcwd(), FILES_IN["sq3"])
    if shred_file and os.path.isfile(path3):
        size = os.path.getsize(path3)
        with open(path3, "r+b") as f:
            f.write(b"0" * size)
            f.flush()
            os.fsync(f.fileno())
        os.remove(path3)
        log(f"  Plik {FILES_IN['sq3']} nadpisany zerami i usuniety z dysku.")
        res["sq3_file_removed"] = True
    elif os.path.isfile(path3):
        log(f"  [UWAGA] Plik {FILES_IN['sq3']} nadal istnieje na dysku (uzyj --shred-sq3 lub usun recznie).")
        res["sq3_file_removed"] = False
    return res


# MODUL 2: Entropia Shannona, min-entropia i rozklad wzorcow m-bitowych

def block_codes(bits, m):
    nb = len(bits) // m
    b = bits[: nb * m].reshape(nb, m)
    codes = np.zeros(nb, dtype=np.int64)
    for k in range(m):
        codes = (codes << 1) | b[:, k]
    return codes, nb


def pattern_stats(bits, m):
    codes, nb = block_codes(bits, m)
    counts = np.bincount(codes, minlength=2 ** m)
    p = counts / nb
    nz = p[p > 0]
    h1 = float(-np.sum(nz * np.log2(nz)) / m)
    hinf = float(-math.log2(p.max()) / m)
    ideal = 1.0 / 2 ** m
    chi2 = float(nb * np.sum((p - ideal) ** 2 / ideal))
    return {
        "m": m,
        "n_blocks": int(nb),
        "p": p,
        "H1_per_bit": h1,
        "Hinf_per_bit": hinf,
        "rmse_vs_ideal": float(np.sqrt(np.mean((p - ideal) ** 2))),
        "max_abs_dev": float(np.max(np.abs(p - ideal))),
        "chi2": chi2,
        "chi2_p_value": float(gammaincc((2 ** m - 1) / 2.0, chi2 / 2.0)),
    }


def run_module2(sq1, sq2):
    hdr("MODUL 2: entropia i rozklad wzorcow m-bitowych")
    out = {}
    log(" m | ciag  | H1/bit   | Hmin/bit | RMSE(ideal) | max|dev|  | p(chi2)  | RMSE(Q1-Q2)")
    for m in (1, 2, 3, 4, 8):
        s1, s2 = pattern_stats(sq1, m), pattern_stats(sq2, m)
        rm = float(np.sqrt(np.mean((s1["p"] - s2["p"]) ** 2)))
        out[m] = {"S_Q1": s1, "S_Q2": s2, "rmse_Q1_vs_Q2": rm}
        for nm, s in (("S_Q1", s1), ("S_Q2", s2)):
            log(f" {m} | {nm} | {s['H1_per_bit']:.6f} | {s['Hinf_per_bit']:.6f} | "
                f"{s['rmse_vs_ideal']:.3e}  | {s['max_abs_dev']:.3e} | {s['chi2_p_value']:.4f}   | "
                f"{rm:.3e}" if nm == "S_Q1" else
                f" {m} | {nm} | {s['H1_per_bit']:.6f} | {s['Hinf_per_bit']:.6f} | "
                f"{s['rmse_vs_ideal']:.3e}  | {s['max_abs_dev']:.3e} | {s['chi2_p_value']:.4f}   |")
    return out


# MODUL 3: Post-processing (Korektor von Neumanna, XOR-folding, SHA-256)

def von_neumann(bits):
    n = len(bits) // 2 * 2
    a, b = bits[:n:2], bits[1:n:2]
    keep = a != b
    return a[keep].copy()            # 01 -> 0, 10 -> 1


def xor_fold(bits):
    n = len(bits) // 2 * 2
    return (bits[0:n:2] ^ bits[1:n:2]).astype(np.uint8)


def sha256_condition(bits):
    nblocks = len(bits) // 512
    if nblocks == 0:
        return np.zeros(0, dtype=np.uint8)
    data = np.packbits(bits[: nblocks * 512]).reshape(nblocks, 64)
    out = bytearray()
    for row in data:
        out += hashlib.sha256(row.tobytes()).digest()
    return np.unpackbits(np.frombuffer(bytes(out), dtype=np.uint8))


def bit_stats(bits):
    n = len(bits)
    if n == 0:
        return {
            "length": 0, "frac_zeros": float("nan"), "H_shannon": float("nan"),
            "H_min": float("nan"), "H8_shannon": float("nan"), "H8_min": float("nan"),
        }
    p0 = float(np.count_nonzero(bits == 0)) / n
    p1 = 1.0 - p0
    h1 = -sum(p * math.log2(p) for p in (p0, p1) if p > 0)
    hmin = -math.log2(max(p0, p1))
    h8 = h8m = float("nan")
    if n >= 8 * 512:
        s8 = pattern_stats(bits, 8)
        h8, h8m = s8["H1_per_bit"], s8["Hinf_per_bit"]
    return {
        "length": n, "frac_zeros": p0, "H_shannon": h1, "H_min": hmin,
        "H8_shannon": h8, "H8_min": h8m,
    }


def run_module3(sq1, sq2):
    hdr("MODUL 3: post-processing (von Neumann, XOR-folding 2:1, SHA-256)")
    processed, table = {}, []
    for nm, s in (("S_Q1", sq1), ("S_Q2", sq2)):
        d = {"raw": s, "vn": von_neumann(s), "xor": xor_fold(s), "sha": sha256_condition(s)}
        processed[nm] = d
        for key, label in (("raw", "surowy"), ("vn", "von Neumann"),
                           ("xor", "XOR-folding 2:1"), ("sha", "SHA-256 (512->256)")):
            st = bit_stats(d[key])
            st.update({"sequence": nm, "method": label})
            table.append(st)
    log(f"{'Ciag':<5} {'Metoda':<20} {'Dlugosc':>9} {'%zer':>8} {'H1':>8} {'Hmin':>8} {'H8/bit':>8} {'Hmin8/bit':>9}")
    for r in table:
        log(f"{r['sequence']:<5} {r['method']:<20} {r['length']:>9d} {100*r['frac_zeros']:>8.3f} "
            f"{r['H_shannon']:>8.5f} {r['H_min']:>8.5f} {r['H8_shannon']:>8.5f} {r['H8_min']:>9.5f}")
    log("\n  Uwagi: von Neumann usuwa bias tylko dla bitow niezaleznych o stalym p; XOR-folding redukuje"
        "\n  bias do ok. 2*eps^2 kosztem polowy dlugosci; SHA-256 daje wyjscie o idealnym balansie"
        "\n  niezaleznie od jakosci wejscia - jego statystyka NIE swiadczy o entropii zrodla."
        "\n  Bezpieczny wsad: min-entropia bloku 512 bit powinna wyraznie przekraczac 256 bit"
        f"\n  (tu ok. {512*bit_stats(sq2)['H_min']:.1f} bit wg estymaty 1-bitowej dla S_Q2).")
    return processed, table


# MODUL 4: 12 testow statystycznych NIST SP 800-22

def _clip(p):
    return float(min(1.0, max(0.0, p)))


def _res(ps, note="", extra=None, applicable=True):
    return {
        "applicable": applicable,
        "ps": {k: _clip(v) for k, v in ps.items()},
        "note": note,
        "extra": extra or {},
    }


def _na(note):
    return {"applicable": False, "ps": {}, "note": note, "extra": {}}


def t_frequency(b):
    n = len(b)
    if n < 100:
        return _na("n < 100")
    s = 2 * int(np.count_nonzero(b)) - n
    return _res({"Frequency": erfc(abs(s) / math.sqrt(n) / math.sqrt(2.0))})


def t_block_frequency(b, M=128):
    n = len(b)
    N = n // M
    if n < 100 or N < 1:
        return _na("n za male")
    blocks = b[: N * M].reshape(N, M)
    pi = blocks.sum(axis=1, dtype=np.int64) / M
    chi2 = 4.0 * M * np.sum((pi - 0.5) ** 2)
    note = f"M=128, N={N}" + (" (zalecenie NIST: N<100)" if N >= 100 else "")
    return _res({"BlockFrequency": gammaincc(N / 2.0, chi2 / 2.0)}, note)


def t_runs(b):
    n = len(b)
    if n < 100:
        return _na("n < 100")
    pi = np.count_nonzero(b) / n
    if abs(pi - 0.5) >= 2.0 / math.sqrt(n):
        return _res({"Runs": 0.0}, "warunek wstepny czestosci niespelniony -> p=0")
    v = 1 + int(np.count_nonzero(b[1:] != b[:-1]))
    p = erfc(abs(v - 2.0 * n * pi * (1 - pi)) / (2.0 * math.sqrt(2.0 * n) * pi * (1 - pi)))
    return _res({"Runs": p})


def t_longest_run(b):
    n = len(b)
    if n < 128:
        return _na("n < 128")
    if n >= 750_000:
        M, lo, hi = 10_000, 10, 16
        pi = np.array([0.0882, 0.2092, 0.2483, 0.1933, 0.1208, 0.0675, 0.0727])
    elif n >= 6272:
        M, lo, hi = 128, 4, 9
        pi = np.array([0.1174, 0.2430, 0.2493, 0.1752, 0.1027, 0.1124])
    else:
        M, lo, hi = 8, 1, 4
        pi = np.array([0.2148, 0.3672, 0.2305, 0.1875])
    N = n // M
    blocks = b[: N * M].reshape(N, M)
    run = np.zeros(N, dtype=np.int64)
    best = np.zeros(N, dtype=np.int64)
    for j in range(M):
        run = (run + 1) * blocks[:, j]
        np.maximum(best, run, out=best)
    idx = np.clip(best, lo, hi) - lo
    v = np.bincount(idx, minlength=len(pi)).astype(np.float64)
    chi2 = np.sum((v - N * pi) ** 2 / (N * pi))
    K = len(pi) - 1
    return _res({"LongestRun": gammaincc(K / 2.0, chi2 / 2.0)}, f"M={M}, N={N}, K={K}")


def t_rank(b):
    n = len(b)
    N = n // 1024
    if N < 38:
        return _na(f"n < 38912 (N={N} macierzy < 38)")
    arr = b[: N * 1024].reshape(N, 32, 32)
    rows = np.zeros((N, 32), dtype=np.int64)
    for k in range(32):
        rows = (rows << 1) | arr[:, :, k]
    used = np.zeros((N, 32), dtype=bool)
    rank = np.zeros(N, dtype=np.int64)
    ar = np.arange(N)
    for c in range(32):
        bit = ((rows >> c) & 1).astype(bool)
        cand = bit & ~used
        has = cand.any(axis=1)
        idx = cand.argmax(axis=1)
        pivot = rows[ar, idx]
        elim = bit & has[:, None]
        elim[ar[has], idx[has]] = False
        rows = np.where(elim, rows ^ pivot[:, None], rows)
        used[ar[has], idx[has]] = True
        rank += has
    f32 = int(np.count_nonzero(rank == 32))
    f31 = int(np.count_nonzero(rank == 31))
    f30 = N - f32 - f31
    pi = np.array([0.2887880951, 0.5775761902, 0.1336357147])    # = 0.2888, 0.5776, 0.1336
    f = np.array([f32, f31, f30], dtype=np.float64)
    chi2 = np.sum((f - N * pi) ** 2 / (N * pi))
    return _res({"Rank": math.exp(-chi2 / 2.0)}, f"N={N}", {"F32": f32, "F31": f31, "Frest": f30})


def t_dft(b):
    n = len(b) // 2 * 2
    if n < 1000:
        return _na("n < 1000")
    x = 2.0 * b[:n] - 1.0
    mod = np.abs(np.fft.rfft(x))[: n // 2]
    T = math.sqrt(math.log(1.0 / 0.05) * n)
    N0 = 0.95 * n / 2.0
    N1 = int(np.count_nonzero(mod < T))
    d = (N1 - N0) / math.sqrt(n * 0.95 * 0.05 / 4.0)
    return _res({"DFT": erfc(abs(d) / math.sqrt(2.0))}, f"N1={N1}, N0={N0:.1f}")


def t_nonoverlapping(b, template=(0, 0, 0, 0, 0, 0, 0, 0, 1), N=8):
    n = len(b)
    m = len(template)
    M = n // N
    if M < 2 * m or n < 1000:
        return _na("n za male")
    tcode = 0
    for t in template:
        tcode = (tcode << 1) | t
    W = np.zeros(N, dtype=np.float64)
    for i in range(N):
        blk = b[i * M:(i + 1) * M].astype(np.int64)
        L = M - m + 1
        code = np.zeros(L, dtype=np.int64)
        for k in range(m):
            code = (code << 1) | blk[k:k + L]
        pos = np.flatnonzero(code == tcode)
        cnt, last = 0, -m
        for p in pos:
            if p >= last + m:
                cnt += 1
                last = int(p)
        W[i] = cnt
    mu = (M - m + 1) / 2.0 ** m
    sigma2 = M * (1.0 / 2.0 ** m - (2 * m - 1) / 2.0 ** (2 * m))
    chi2 = np.sum((W - mu) ** 2 / sigma2)
    return _res({"NonOverlapping": gammaincc(N / 2.0, chi2 / 2.0)}, f"m={m}, N={N}, M={M}")


def t_overlapping(b, m=9, M=1032):
    n = len(b)
    N = n // M
    pi = np.array([0.364091, 0.185659, 0.139381, 0.100571, 0.0704323, 0.139865])
    if N * pi.min() < 5:
        return _na(f"N={N} za male (N*pi_min < 5)")
    blocks = b[: N * M].reshape(N, M).astype(np.int64)
    c = np.zeros((N, M + 1), dtype=np.int64)
    np.cumsum(blocks, axis=1, out=c[:, 1:])
    w = c[:, m:] - c[:, :-m]
    occ = np.minimum((w == m).sum(axis=1), 5)
    v = np.bincount(occ, minlength=6).astype(np.float64)
    chi2 = np.sum((v - N * pi) ** 2 / (N * pi))
    return _res({"Overlapping": gammaincc(5 / 2.0, chi2 / 2.0)}, f"m={m}, M={M}, N={N}")


def _cyc_counts(bits, m):
    n = len(bits)
    ext = np.concatenate((bits, bits[: m - 1])).astype(np.int64)
    codes = np.zeros(n, dtype=np.int64)
    for k in range(m):
        codes = (codes << 1) | ext[k:k + n]
    return np.bincount(codes, minlength=2 ** m)


def _psi2(bits, m):
    if m <= 0:
        return 0.0
    n = len(bits)
    c = _cyc_counts(bits, m).astype(np.float64)
    return (2.0 ** m / n) * float(np.sum(c ** 2)) - n


def t_serial(b):
    n = len(b)
    m = min(8, int(math.floor(math.log2(n))) - 3) if n > 16 else 0
    if m < 3:
        return _na("n za male dla testu serial")
    p1, p2, p3 = _psi2(b, m), _psi2(b, m - 1), _psi2(b, m - 2)
    d1, d2 = p1 - p2, p1 - 2 * p2 + p3
    return _res(
        {"Serial-p1": gammaincc(2.0 ** (m - 2), d1 / 2.0),
         "Serial-p2": gammaincc(2.0 ** (m - 3), d2 / 2.0)},
        f"m={m}" + ("" if m == 8 else " (zredukowane wg NIST: m < log2(n)-2)"),
    )


def _phi(bits, m):
    n = len(bits)
    c = _cyc_counts(bits, m).astype(np.float64) / n
    c = c[c > 0]
    return float(np.sum(c * np.log(c)))


def t_apen(b):
    n = len(b)
    m = min(8, int(math.floor(math.log2(n))) - 6) if n > 128 else 0
    if m < 2:
        return _na("n za male dla ApEn")
    ap = _phi(b, m) - _phi(b, m + 1)
    chi2 = 2.0 * n * (math.log(2.0) - ap)
    return _res(
        {"ApEn": gammaincc(2.0 ** (m - 1), chi2 / 2.0)},
        f"m={m}" + ("" if m == 8 else " (zredukowane wg NIST: m < log2(n)-5)"),
    )


def _cusum_p(z, n):
    if z <= 0:
        return 1.0
    sq = math.sqrt(n)
    k1 = np.arange(int(math.floor((-n / z + 1) / 4)), int(math.floor((n / z - 1) / 4)) + 1)
    s1 = np.sum(norm.cdf((4 * k1 + 1) * z / sq) - norm.cdf((4 * k1 - 1) * z / sq))
    k2 = np.arange(int(math.floor((-n / z - 3) / 4)), int(math.floor((n / z - 1) / 4)) + 1)
    s2 = np.sum(norm.cdf((4 * k2 + 3) * z / sq) - norm.cdf((4 * k2 + 1) * z / sq))
    return 1.0 - float(s1) + float(s2)


def t_cusum(b):
    n = len(b)
    if n < 100:
        return _na("n < 100")
    x = 2 * b.astype(np.int64) - 1
    zf = int(np.max(np.abs(np.cumsum(x))))
    zb = int(np.max(np.abs(np.cumsum(x[::-1]))))
    return _res({"Cusum-F": _cusum_p(zf, n), "Cusum-B": _cusum_p(zb, n)}, f"zF={zf}, zB={zb}")


def t_rev(b):
    n = len(b)
    x = 2 * b.astype(np.int64) - 1
    s = np.concatenate(([0], np.cumsum(x), [0]))
    J = int(np.count_nonzero(s == 0)) - 1
    if J < 1:
        return _na("J = 0 cykli")
    per_state = {}
    for st in range(-9, 10):
        if st == 0:
            continue
        xi = int(np.count_nonzero(s == st))
        per_state[st] = float(erfc(abs(xi - J) / math.sqrt(2.0 * J * (4 * abs(st) - 2))))
    pmin = min(per_state.values())
    ok = J >= 500
    return _res(
        {"RandExcVar-min": pmin},
        f"J={J}" + ("" if ok else " (< 500: test nie ma zastosowania wg NIST)"),
        {"J": J, "per_state": per_state},
        applicable=ok,
    )


TESTS = [
    ("1 Frequency (Monobit)", t_frequency),
    ("2 Block Frequency", t_block_frequency),
    ("3 Runs", t_runs),
    ("4 Longest Run of Ones", t_longest_run),
    ("5 Binary Matrix Rank", t_rank),
    ("6 DFT (Spectral)", t_dft),
    ("7 Non-overlapping Template", t_nonoverlapping),
    ("8 Overlapping Template", t_overlapping),
    ("9 Serial", t_serial),
    ("10 Approximate Entropy", t_apen),
    ("11 Cumulative Sums", t_cusum),
    ("12 Random Excursions Variant", t_rev),
]


def run_battery(bits):
    out = {}
    for name, fn in TESTS:
        try:
            out[name] = fn(bits)
        except Exception as e:  # noqa
            out[name] = _na(f"blad wykonania: {e}")
    return out


def status(entry, p):
    if not entry["applicable"]:
        return "N/D"
    return "PASS" if p >= ALPHA else "FAIL"


def print_battery(title, res, n):
    log(f"\n--- {title} (n = {n} bitow) ---")
    npass = ntot = 0
    for name, e in res.items():
        if not e["ps"]:
            log(f"  {name:<30} {'-':>10}   N/D   {e['note']}")
            continue
        for lab, p in e["ps"].items():
            st = status(e, p)
            if st != "N/D":
                ntot += 1
                npass += st == "PASS"
            log(f"  {name:<30} {lab:<15} p = {p:.6f}  {st:<5} {e['note']}")
    log(f"  Podsumowanie: zaliczone {npass}/{ntot} p-wartosci (alpha = {ALPHA})")
    return npass, ntot


# MODUL 5: Certyfikacja pojedynczych ciagow oraz analiza wielosekwencyjna

def multi_sequence(bits, K, n_sub):
    if len(bits) < K * n_sub:
        n_sub = len(bits) // K
    seqs = bits[: K * n_sub].reshape(K, n_sub)
    coll = {}
    for i in range(K):
        res = run_battery(np.ascontiguousarray(seqs[i]))
        for name, e in res.items():
            if not e["applicable"]:
                continue
            for lab, p in e["ps"].items():
                coll.setdefault((name, lab), []).append(p)
    out = {}
    p_hat = 1.0 - ALPHA
    for (name, lab), ps in coll.items():
        ps = np.array(ps)
        Ke = len(ps)
        if Ke < 50:
            out[lab] = {"test": name, "K_effective": Ke, "applicable": False}
            continue
        prop = float(np.mean(ps >= ALPHA))
        low = p_hat - 3.0 * math.sqrt(p_hat * (1 - p_hat) / Ke)
        hist, _ = np.histogram(ps, bins=10, range=(0.0, 1.0))
        exp = Ke / 10.0
        chi2 = float(np.sum((hist - exp) ** 2 / exp))
        pu = float(gammaincc(9 / 2.0, chi2 / 2.0))
        out[lab] = {
            "test": name, "K_effective": Ke, "n_sub": n_sub, "applicable": True,
            "proportion": prop, "prop_low": low, "prop_ok": prop >= low,
            "hist": hist.tolist(), "uniformity_p": pu, "uniformity_ok": pu >= 0.0001,
        }
    return out, n_sub


def print_multi(title, out, n_sub):
    log(f"\n--- Analiza wielosekwencyjna: {title} (K={K_SEQ} x {n_sub} bitow) ---")
    log(f"  {'Test':<16} {'K_eff':>5} {'Proporcja':>10} {'Dolna gr.':>10} {'Wynik':>6} {'p(jedn.)':>9} {'Wynik':>6}")
    for lab, r in out.items():
        if not r["applicable"]:
            log(f"  {lab:<16} {r['K_effective']:>5}   N/D (za malo zastosowalnych podciagow)")
            continue
        log(f"  {lab:<16} {r['K_effective']:>5} {r['proportion']:>10.4f} {r['prop_low']:>10.4f} "
            f"{'PASS' if r['prop_ok'] else 'FAIL':>6} {r['uniformity_p']:>9.4f} "
            f"{'PASS' if r['uniformity_ok'] else 'FAIL':>6}")


def run_module5(processed):
    hdr("MODUL 4/5: bateria 12 testow NIST SP 800-22 + analiza wielosekwencyjna")
    q1, q2 = processed["S_Q1"], processed["S_Q2"]
    datasets = [
        ("S_Q1 surowy", q1["raw"][:N_RAW_SINGLE]),
        ("S_Q2 surowy", q2["raw"][:N_RAW_SINGLE]),
        ("S_Q1 Von Neumann", q1["vn"]),
        ("S_Q2 Von Neumann", q2["vn"]),
        ("S_Q1 XOR-folding", q1["xor"]),
        ("S_Q2 XOR-folding", q2["xor"]),
    ]
    single, summary = {}, {}
    for name, bits in datasets:
        t = time.time()
        res = run_battery(bits)
        single[name] = {"n_bits": len(bits), "results": res}
        npass, ntot = print_battery(name, res, len(bits))
        summary[name] = {"passed": int(npass), "total": int(ntot)}
        log(f"  (czas: {time.time() - t:.1f} s)")

    multi = {}
    for name, bits, nsub in (
        ("S_Q1 surowy", q1["raw"], N_SUB_RAW),
        ("S_Q2 surowy", q2["raw"], N_SUB_RAW),
        ("S_Q1 XOR-folding", q1["xor"], N_SUB_XOR),
        ("S_Q2 XOR-folding", q2["xor"], N_SUB_XOR),
    ):
        t = time.time()
        out, used = multi_sequence(bits, K_SEQ, nsub)
        multi[name] = {"n_sub": used, "results": out}
        print_multi(name, out, used)
        log(f"  (czas: {time.time() - t:.1f} s)")
    return single, multi, summary, datasets


# MODUL 6: Wizualizacja, tabele CSV/LaTeX/JSON i eksport klucza dla PQC/AES

def acf(x, maxlag):
    x = x - x.mean()
    d = float(np.dot(x, x))
    return np.array([float(np.dot(x[:-k], x[k:])) / d for k in range(1, maxlag + 1)])


def ccf(x1, x2, maxlag):
    x1, x2 = x1 - x1.mean(), x2 - x2.mean()
    n = min(len(x1), len(x2))
    x1, x2 = x1[:n], x2[:n]
    d = math.sqrt(float(np.dot(x1, x1)) * float(np.dot(x2, x2)))
    lags = np.arange(-maxlag, maxlag + 1)
    vals = []
    for L in lags:
        if L >= 0:
            vals.append(float(np.dot(x1[: n - L], x2[L:])) / d)
        else:
            vals.append(float(np.dot(x1[-L:], x2[: n + L])) / d)
    return lags, np.array(vals)


def plot_patterns(processed, fname):
    fig, axes = plt.subplots(3, 2, figsize=(14, 11))
    methods = [("raw", "surowy"), ("vn", "Von Neumann"), ("xor", "XOR-folding")]
    for r, m in enumerate((1, 2, 3)):
        labels = [format(i, f"0{m}b") for i in range(2 ** m)]
        x = np.arange(2 ** m)
        for c, nm in enumerate(("S_Q1", "S_Q2")):
            ax = axes[r, c]
            w = 0.26
            for j, (key, lab) in enumerate(methods):
                bits = processed[nm][key]
                if len(bits) >= m * 100:
                    p = pattern_stats(bits, m)["p"]
                    ax.bar(x + (j - 1) * w, p, w, label=lab)
            ax.axhline(1.0 / 2 ** m, color="k", ls="--", lw=1, label=f"idealny 1/{2 ** m}")
            ax.set_xticks(x)
            ax.set_xticklabels(labels)
            ax.set_title(f"{nm} - rozklad wzorcow {m}-bitowych")
            ax.set_xlabel("Wzorzec")
            ax.set_ylabel("Czestosc wzgledna")
            ax.legend(fontsize=8)
    fig.suptitle("EQRNG Etap 2: rozklad wzorcow przed i po debiasingu", fontsize=14)
    fig.tight_layout()
    fig.savefig(fname, dpi=DPI)
    plt.close(fig)


def plot_pvalues(single, fname):
    labels = []
    for ds in single.values():
        for e in ds["results"].values():
            if e["applicable"]:
                for lab in e["ps"]:
                    if lab not in labels:
                        labels.append(lab)
    markers = {"surowy": "o", "Von Neumann": "s", "XOR-folding": "^"}
    colors = {"S_Q1": "tab:blue", "S_Q2": "tab:orange"}
    fig, ax = plt.subplots(figsize=(14, 6.5))
    names = list(single.keys())
    for k, nm in enumerate(names):
        seq = nm.split()[0]
        proc = nm.split(" ", 1)[1]
        xs, ys = [], []
        for i, lab in enumerate(labels):
            for e in single[nm]["results"].values():
                if e["applicable"] and lab in e["ps"]:
                    xs.append(i + (k - 2.5) * 0.12)
                    ys.append(max(e["ps"][lab], 1e-6))
        ax.scatter(xs, ys, marker=markers[proc], color=colors[seq], s=55,
                   edgecolor="k", linewidth=0.4, label=nm)
    ax.axhline(ALPHA, color="red", ls="--", lw=1.5, label=f"prog alpha = {ALPHA}")
    ax.set_yscale("log")
    ax.set_ylim(5e-7, 1.5)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_ylabel("Wartosc p (skala logarytmiczna)")
    ax.set_title("Testy NIST SP 800-22: wartosci p dla S_Q1 i S_Q2 (p < 1e-6 przyciete do 1e-6)")
    ax.grid(True, which="both", alpha=0.25)
    ax.legend(fontsize=8, ncol=2, loc="lower right")
    fig.tight_layout()
    fig.savefig(fname, dpi=DPI)
    plt.close(fig)


def plot_autocorr(sq1, sq2, fname):
    n = min(len(sq1), len(sq2))
    x1 = 2.0 * sq1[:n] - 1.0
    x2 = 2.0 * sq2[:n] - 1.0
    maxlag = 50
    a1, a2 = acf(x1, maxlag), acf(x2, maxlag)
    lags, cc = ccf(x1, x2, maxlag)
    band = 1.96 / math.sqrt(n)
    fig, axes = plt.subplots(2, 1, figsize=(12, 8.5))
    k = np.arange(1, maxlag + 1)
    ax = axes[0]
    ax.bar(k - 0.2, a1, 0.4, label="S_Q1")
    ax.bar(k + 0.2, a2, 0.4, label="S_Q2")
    ax.axhline(band, color="r", ls="--", lw=1, label="95% przedzial dla szumu bialego")
    ax.axhline(-band, color="r", ls="--", lw=1)
    ax.set_xlabel("Opoznienie (lag)")
    ax.set_ylabel("Autokorelacja")
    ax.set_title("Funkcja autokorelacji, lag 1-50")
    ax.legend()
    ax = axes[1]
    ax.bar(lags, cc, 0.8, color="tab:green", label="korelacja wzajemna S_Q1 vs S_Q2")
    ax.axhline(band, color="r", ls="--", lw=1, label="95% przedzial")
    ax.axhline(-band, color="r", ls="--", lw=1)
    ax.set_xlabel("Opoznienie (lag)")
    ax.set_ylabel("Korelacja wzajemna")
    ax.set_title("Korelacja wzajemna S_Q1 vs S_Q2, lag -50..50")
    ax.legend()
    fig.tight_layout()
    fig.savefig(fname, dpi=DPI)
    plt.close(fig)
    return {
        "acf_S_Q1": a1.tolist(), "acf_S_Q2": a2.tolist(),
        "ccf_lags": lags.tolist(), "ccf": cc.tolist(), "band_95": band,
        "acf_S_Q1_outside_band": int(np.sum(np.abs(a1) > band)),
        "acf_S_Q2_outside_band": int(np.sum(np.abs(a2) > band)),
        "ccf_outside_band": int(np.sum(np.abs(cc) > band)),
    }


def write_csv(single, multi, fname):
    fields = [
        "typ", "zbior", "test", "etykieta", "n_bits", "p_value", "alpha", "wynik", "uwagi",
        "K_eff", "n_sub", "proporcja", "dolna_granica", "wynik_proporcji",
        "p_jednorodnosci", "wynik_jednorodnosci",
    ]
    with open(fname, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=fields, restval="")
        w.writeheader()
        for ds, d in single.items():
            for name, e in d["results"].items():
                if not e["ps"]:
                    w.writerow({
                        "typ": "pojedynczy", "zbior": ds, "test": name,
                        "n_bits": d["n_bits"], "wynik": "N/D", "uwagi": e["note"],
                    })
                for lab, p in e["ps"].items():
                    w.writerow({
                        "typ": "pojedynczy", "zbior": ds, "test": name, "etykieta": lab,
                        "n_bits": d["n_bits"], "p_value": f"{p:.8g}", "alpha": ALPHA,
                        "wynik": status(e, p), "uwagi": e["note"],
                    })
        for ds, d in multi.items():
            for lab, r in d["results"].items():
                row = {
                    "typ": "wielosekwencyjny", "zbior": ds, "test": r["test"],
                    "etykieta": lab, "alpha": ALPHA, "K_eff": r["K_effective"],
                }
                if r["applicable"]:
                    row.update({
                        "n_sub": r["n_sub"], "proporcja": f"{r['proportion']:.4f}",
                        "dolna_granica": f"{r['prop_low']:.4f}",
                        "wynik_proporcji": "PASS" if r["prop_ok"] else "FAIL",
                        "p_jednorodnosci": f"{r['uniformity_p']:.6g}",
                        "wynik_jednorodnosci": "PASS" if r["uniformity_ok"] else "FAIL",
                    })
                else:
                    row["wynik"] = "N/D"
                w.writerow(row)


def write_latex(single, fname):
    names = list(single.keys())
    labels = []
    for ds in single.values():
        for e in ds["results"].values():
            for lab in e["ps"]:
                if lab not in labels:
                    labels.append(lab)
    head = {
        "S_Q1 surowy": "$S_{Q_1}$ surowy", "S_Q2 surowy": "$S_{Q_2}$ surowy",
        "S_Q1 Von Neumann": "$S_{Q_1}$ vN", "S_Q2 Von Neumann": "$S_{Q_2}$ vN",
        "S_Q1 XOR-folding": "$S_{Q_1}$ XOR", "S_Q2 XOR-folding": "$S_{Q_2}$ XOR",
    }
    lines = [
        "% Wymaga pakietu booktabs",
        "\\begin{table}[htbp]", "\\centering", "\\small",
        "\\begin{tabular}{l" + "r" * len(names) + "}", "\\toprule",
        "Test & " + " & ".join(head[n] for n in names) + " \\\\", "\\midrule",
    ]
    for lab in labels:
        cells = []
        for nm in names:
            val = "--"
            for e in single[nm]["results"].values():
                if lab in e["ps"] and e["applicable"]:
                    p = e["ps"][lab]
                    val = f"{p:.4f}"
                    if p < ALPHA:
                        val = "\\textbf{" + val + "}"
            cells.append(val)
        lines.append(lab + " & " + " & ".join(cells) + " \\\\")
    lines += [
        "\\bottomrule", "\\end{tabular}",
        "\\caption{Wyniki testów NIST SP 800-22 (wartości $p$; pogrubione: $p<0{,}01$; "
        "-- : test nie ma zastosowania dla danej długości ciągu).}",
        "\\label{tab:eqrng_nist}", "\\end{table}",
    ]
    with open(fname, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")


def export_key(processed):
    key_bits = sha256_condition(processed["S_Q2"]["raw"])
    if len(key_bits) < 512:
        raise ValueError("Za malo bitow do wyeksportowania ziarna 512-bitowego.")
    key_bytes = np.packbits(key_bits).tobytes()
    with open("eqrng_certified_key.bin", "wb") as f:
        f.write(key_bytes)
    seed_hex = key_bytes[:64].hex()
    with open("eqrng_seed_pqc.hex", "w", encoding="utf-8") as f:
        f.write(seed_hex + "\n")
    return {
        "key_bytes": len(key_bytes),
        "key_sha256_of_file": hashlib.sha256(key_bytes).hexdigest(),
        "seed_hex_length_chars": len(seed_hex),
    }


DISCLAIMER = (
    "Nazwa pliku 'certified' wynika ze specyfikacji eksperymentu. Wynik testow statystycznych "
    "(NIST SP 800-22) oznacza jedynie brak wykrytych odstepstw od losowosci; nie jest dowodem "
    "bezpieczenstwa ani nie certyfikuje entropii. Plik zawiera wyjscie SHA-256 z S_Q2, wiec jego "
    "wlasne statystyki nie swiadcza o jakosci zrodla. Dostawca chmury ma dostep do surowych wynikow "
    "pomiarow, wiec klucz nie jest tajny wzgledem dostawcy. Do zastosowan produkcyjnych kluczy "
    "uzywaj sprawdzonego CSPRNG/OS RNG lub zmieszaj to ziarno z takim zrodlem."
)


def run_module6(processed, single, multi, summary, m1, m2, m3, sq1, sq2):
    hdr("MODUL 6: wykresy, raporty, klucz")
    plot_patterns(processed, "eqrng_stage2_patterns.png")
    log("  zapisano eqrng_stage2_patterns.png")
    plot_pvalues(single, "eqrng_stage2_nist_pvalues.png")
    log("  zapisano eqrng_stage2_nist_pvalues.png")
    corr = plot_autocorr(sq1, sq2, "eqrng_stage2_autocorr.png")
    log("  zapisano eqrng_stage2_autocorr.png")
    write_csv(single, multi, "eqrng_stage2_results.csv")
    log("  zapisano eqrng_stage2_results.csv")
    write_latex(single, "eqrng_stage2_table.tex")
    log("  zapisano eqrng_stage2_table.tex")
    key_info = export_key(processed)
    log("  zapisano eqrng_certified_key.bin oraz eqrng_seed_pqc.hex (64 bajty = 512 bitow)")

    m2_json = {
        str(m): {
            "S_Q1": {k: v for k, v in d["S_Q1"].items() if k != "p"} | ({"p": d["S_Q1"]["p"]} if m <= 4 else {}),
            "S_Q2": {k: v for k, v in d["S_Q2"].items() if k != "p"} | ({"p": d["S_Q2"]["p"]} if m <= 4 else {}),
            "rmse_Q1_vs_Q2": d["rmse_Q1_vs_Q2"],
        }
        for m, d in m2.items()
    }
    report = {
        "meta": {
            "generated": datetime.datetime.now().isoformat(timespec="seconds"),
            "alpha": ALPHA, "runtime_s": round(time.time() - T0, 1),
        },
        "module1": m1, "module2": m2_json, "module3_table": m3,
        "nist_single": single, "nist_summary": summary, "nist_multi": multi,
        "correlation": corr, "key_export": key_info, "disclaimer": DISCLAIMER,
    }
    with open("eqrng_stage2_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2, default=json_default)
    log("  zapisano eqrng_stage2_report.json")
    log("\n  UWAGA: " + DISCLAIMER)


# Funkcja glowna
def main():
    args = sys.argv[1:]
    demo = "--demo" in args
    shred = "--shred-sq3" in args
    hdr("EQRNG - ETAP 2: walidacja, post-processing i certyfikacja statystyczna")
    if demo:
        log("[TRYB DEMO] Dane syntetyczne z biasem jak w opisie (S_Q1 ~49.73% zer, S_Q2 ~52.91% zer).")
        rng = np.random.default_rng(12345)
        n = 1_212_209
        sq1 = (rng.random(n) >= 0.4973).astype(np.uint8)
        sq2 = (rng.random(n) >= 0.5291).astype(np.uint8)
        holder = {"sq3": (sq1 ^ sq2).astype(np.uint8)}
        ckpt = None
    else:
        sq1 = load_bits(FILES_IN["sq1"])
        sq2 = load_bits(FILES_IN["sq2"])
        holder = {"sq3": load_bits(FILES_IN["sq3"])}
        ckpt = load_checkpoint(FILES_IN["ckpt"])
    log(f"Wczytano: |S_Q1| = {len(sq1)}, |S_Q2| = {len(sq2)}, |S_Q3| = {len(holder['sq3'])}")

    m1 = run_module1(sq1, sq2, holder, ckpt, shred and not demo)
    m2 = run_module2(sq1, sq2)
    processed, m3 = run_module3(sq1, sq2)
    single, multi, summary, _ = run_module5(processed)
    run_module6(processed, single, multi, summary, m1, m2, m3, sq1, sq2)
    hdr(f"GOTOWE. Calkowity czas: {time.time() - T0:.1f} s")


if __name__ == "__main__":
    main()


EQRNG - ETAP 2: walidacja, post-processing i certyfikacja statystyczna
Wczytano: |S_Q1| = 1212209, |S_Q2| = 1212209, |S_Q3| = 1212209

MODUL 1: weryfikacja regul XOR, niezaleznosc bitowa, kasowanie S_Q3
[checkpoint] klucze liczbowe: ['completed_batches', 'total_batches', 'batch_shots', 'shots_processed', 'valid_shots', 'error_shots', 'counts.000', 'counts.011', 'counts.101', 'counts.110', 'counts.001', 'counts.010']
[checkpoint] poprawne = 1212209 vs dlugosc S_Q1 = 1212209 -> OK
[checkpoint] QBER przeliczony = 6.7532% (wszystkie=1300000)
  Regula S_Q1 ^ S_Q2 == S_Q3: niezgodnosci = 0  -> OK
  Regula S_Q1 ^ S_Q3 == S_Q2: niezgodnosci = 0  -> OK
  Regula S_Q2 ^ S_Q3 == S_Q1: niezgodnosci = 0  -> OK

  Para            | Hamming | Pearson r  | I(X;Y) [bit] | p(niezal.)
  S_Q1 vs S_Q2    | 0.48327 | +0.033839 | 8.262e-04  | 0.0000
  S_Q1 vs S_Q3    | 0.47092 | +0.058378 | 2.460e-03  | 0.0000
  S_Q2 vs S_Q3    | 0.50270 | -0.007366 | 3.914e-05  | 0.0000
  Uwaga: parami ciagi sa niezalezne, 